# Data Preprocessing — Feature Engineering & Transformation
Input: cleaned train / test / store. Output: `preprocessed_train.csv`, `preprocessed_test.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Colab: files uploaded next to the notebook. Repo: ../dataset/cleaned
if Path('cleaned_train.csv').exists():
    DATA = OUT = Path('.')
else:
    DATA, OUT = Path('../dataset/cleaned'), Path('../dataset/preprocessed')
    OUT.mkdir(exist_ok=True)

train_path = next(p for p in [DATA/'cleaned_train.csv', DATA/'cleaned_train.zip'] if p.exists())
opts = dict(parse_dates=['Date'], dtype={'StateHoliday': str})
train = pd.read_csv(train_path, **opts)
test  = pd.read_csv(DATA/'cleaned_test.csv', **opts)
store = pd.read_csv(DATA/'cleaned_store.csv')
print(train.shape, test.shape, store.shape)

(1017155, 9) (41088, 8) (1115, 10)


## 1. Store-level dates (competition / Promo2 start)

In [2]:
store['CompetitionStart'] = [
    pd.Timestamp(int(y), int(m) or 1, 1) if y > 0 else pd.NaT
    for y, m in zip(store['CompetitionOpenSinceYear'], store['CompetitionOpenSinceMonth'])
]
store['Promo2Start'] = [
    pd.Timestamp.fromisocalendar(int(y), int(w), 1) if y > 0 else pd.NaT
    for y, w in zip(store['Promo2SinceYear'], store['Promo2SinceWeek'])
]
store[['Store', 'CompetitionStart', 'Promo2Start']].dropna(thresh=2).head()

,Store,CompetitionStart,Promo2Start
0,1,2008-09-01,NaT
1,2,2007-11-01,2010-03-29
2,3,2006-12-01,2011-04-04
3,4,2009-09-01,NaT
4,5,2015-04-01,NaT


## 2. Merge and time-based features

In [3]:
df = pd.concat([train.assign(Set='train'), test.assign(Set='test')], ignore_index=True)
df = df.merge(store, on='Store', how='left')
assert len(df) == len(train) + len(test)

d = df['Date']
df['Year'] = d.dt.year
df['Month'] = d.dt.month
df['Day'] = d.dt.day
df['WeekOfYear'] = d.dt.isocalendar().week.astype(int)
df['Quarter'] = d.dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)
df['Season'] = df['Month'].map({12: 'winter', 1: 'winter', 2: 'winter',
                                3: 'spring', 4: 'spring', 5: 'spring',
                                6: 'summer', 7: 'summer', 8: 'summer',
                                9: 'autumn', 10: 'autumn', 11: 'autumn'})
df[['Date', 'Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Season']].head()

,Date,Year,Month,Day,WeekOfYear,Quarter,IsWeekend,Season
0,2013-01-01,2013,1,1,1,1,0,winter
1,2013-01-01,2013,1,1,1,1,0,winter
2,2013-01-01,2013,1,1,1,1,0,winter
3,2013-01-01,2013,1,1,1,1,0,winter
4,2013-01-01,2013,1,1,1,1,0,winter


## 3. Promotion and competition features

In [4]:
def months_between(a, b):
    return (a.dt.year - b.dt.year) * 12 + (a.dt.month - b.dt.month)

df['CompetitionOpenMonths'] = months_between(df['Date'], df['CompetitionStart']).clip(lower=0).fillna(0)
df['Promo2OpenMonths'] = months_between(df['Date'], df['Promo2Start']).clip(lower=0).fillna(0)

abbr = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
        7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
in_interval = np.fromiter((abbr[m] in iv for m, iv in zip(df['Month'], df['PromoInterval'])), bool, len(df))
df['IsPromo2Active'] = ((df['Promo2'] == 1) & (df['Date'] >= df['Promo2Start']) & in_interval).astype(int)

df[['Promo', 'Promo2', 'IsPromo2Active']].mean().round(3)

Promo             0.382
Promo2            0.504
IsPromo2Active    0.149
dtype: float64

## 4. Categorical encoding
One-hot for low-cardinality columns. `Store` stays a numeric ID (1,115 values).

In [5]:
cats = {'StateHoliday': ['0', 'a', 'b', 'c'], 'StoreType': list('abcd'),
        'Assortment': list('abc'), 'Season': ['autumn', 'spring', 'summer', 'winter']}
for col, levels in cats.items():
    df[col] = pd.Categorical(df[col].astype(str), categories=levels)

df = pd.get_dummies(df, columns=list(cats), drop_first=True, dtype=int)
[c for c in df.columns if c.startswith(('StateHoliday_', 'StoreType_', 'Assortment_', 'Season_'))]

['StateHoliday_a',
 'StateHoliday_b',
 'StateHoliday_c',
 'StoreType_b',
 'StoreType_c',
 'StoreType_d',
 'Assortment_b',
 'Assortment_c',
 'Season_spring',
 'Season_summer',
 'Season_winter']

## 5. Transformations and scaling
- `Sales`: log1p target added as `SalesLog` (right-skewed).
- `CompetitionDistance`: log1p.
- Standardize only the 3 continuous features, fitted on train only. Binary/one-hot/ID/date parts stay as they are.

In [6]:
df['SalesLog'] = np.log1p(df['Sales'])
df['CompetitionDistanceLog'] = np.log1p(df['CompetitionDistance'])

scale_cols = ['CompetitionDistanceLog', 'CompetitionOpenMonths', 'Promo2OpenMonths']
is_train = df['Set'] == 'train'
mu, sd = df.loc[is_train, scale_cols].mean(), df.loc[is_train, scale_cols].std()
for c in scale_cols:
    df[c + '_scaled'] = (df[c] - mu[c]) / sd[c]

df[[c + '_scaled' for c in scale_cols]].loc[is_train].describe().loc[['mean', 'std']].round(3)

,CompetitionDistanceLog_scaled,CompetitionOpenMonths_scaled,Promo2OpenMonths_scaled
mean,-0.0,-0.0,0.0
std,1.0,1.0,1.0


## 6. Final columns and split

In [7]:
drop = ['Customers', 'CompetitionDistance', 'CompetitionDistanceLog', 'CompetitionOpenMonths', 'Promo2OpenMonths',
        'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear',
        'PromoInterval', 'CompetitionStart', 'Promo2Start']
df = df.drop(columns=drop)

num = df.select_dtypes('float').columns.drop(['Sales', 'SalesLog', 'Id'], errors='ignore')
df[num] = df[num].round(6)
df['SalesLog'] = df['SalesLog'].round(6)

train_f = df[df['Set'] == 'train'].drop(columns=['Set', 'Id']).reset_index(drop=True)
test_f = df[df['Set'] == 'test'].drop(columns=['Set', 'Sales', 'SalesLog']).reset_index(drop=True)
test_f['Id'] = test_f['Id'].astype(int)
test_f = test_f[['Id'] + [c for c in test_f.columns if c != 'Id']]
print(train_f.shape, test_f.shape)
train_f.head()

(1017155, 30) (41088, 29)


,Store,DayOfWeek,Date,Sales,Open,Promo,SchoolHoliday,Promo2,Year,Month,...,StoreType_d,Assortment_b,Assortment_c,Season_spring,Season_summer,Season_winter,SalesLog,CompetitionDistanceLog_scaled,CompetitionOpenMonths_scaled,Promo2OpenMonths_scaled
0,1,2,2013-01-01,0.0,0,0,1,0,2013,1,...,0,0,0,0,0,1,0.0,-0.319081,0.201945,-0.690759
1,2,2,2013-01-01,0.0,0,0,1,1,2013,1,...,0,0,0,0,0,1,0.0,-0.833170,0.385855,1.097573
2,3,2,2013-01-01,0.0,0,0,1,1,2013,1,...,0,0,0,0,0,1,0.0,1.228361,0.588155,0.413799
3,4,2,2013-01-01,0.0,0,0,1,0,2013,1,...,0,0,1,0,0,1,0.0,-0.779240,-0.018746,-0.690759
4,5,2,2013-01-01,0.0,0,0,1,0,2013,1,...,0,0,0,0,0,1,0.0,1.710124,-0.754385,-0.690759


## 7. Final dataset check

In [8]:
feats = [c for c in train_f.columns if c not in ('Sales', 'SalesLog')]
assert train_f.isna().sum().sum() == 0 and test_f.isna().sum().sum() == 0
assert np.isfinite(train_f.select_dtypes('number')).all().all() and np.isfinite(test_f.select_dtypes('number')).all().all()
assert set(feats) == set(test_f.columns) - {'Id'}
assert not train_f.duplicated(['Store', 'Date']).any() and not test_f.duplicated(['Store', 'Date']).any()
assert pd.api.types.is_datetime64_any_dtype(train_f['Date'])
assert len(train_f) == len(train) and len(test_f) == len(test)
print('OK')
train_f.dtypes

OK


Store                                     int64
DayOfWeek                                 int64
Date                             datetime64[us]
Sales                                   float64
Open                                      int64
Promo                                     int64
SchoolHoliday                             int64
Promo2                                    int64
Year                                      int32
Month                                     int32
Day                                       int32
WeekOfYear                                int64
Quarter                                   int32
IsWeekend                                 int64
IsPromo2Active                            int64
StateHoliday_a                            int64
StateHoliday_b                            int64
StateHoliday_c                            int64
StoreType_b                               int64
StoreType_c                               int64
StoreType_d                             

## 8. Save

In [9]:
train_f.to_csv(OUT / 'preprocessed_train.csv', index=False)
test_f.to_csv(OUT / 'preprocessed_test.csv', index=False)
print('Saved to', OUT.resolve())

Saved to /home/claude/repo/dataset/preprocessed
